# Marketing Campaign Performance: Exploratory Data Analysis

**Goal:** understand how advertising drives user acquisition and engagement, and tell the story the data contains.

**Dataset:** [Marketing Campaign Performance (Kaggle)](https://www.kaggle.com/datasets/manishabhatt22/marketing-campaign-performance-dataset)

**Key questions**
1. Which channels acquire users most efficiently (CTR, conversion rate, cost per conversion, ROI)?
2. Do more impressions or higher spend translate into more engagement?
3. Which customer segments, audiences, countries and languages respond best?
4. Does campaign duration affect ROI or engagement?
5. Is there seasonality or a trend over time?
6. Which combinations of channel, campaign type and audience perform best, and which waste budget?

**Limitations:** the dataset looks synthetic and has no retention data. `Engagement_Score` is the closest proxy.

## 1. Setup and data loading

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# The notebook lives in notebooks/, so the data folder is one level up
df = pd.read_csv("../data/marketing_campaign_dataset.csv")

## 2. First inspection

Goal: understand the structure before changing anything (size, column types, nulls, duplicates, value ranges).

In [3]:
# Display the information about the DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  str    
 2   Campaign_Type     200000 non-null  str    
 3   Target_Audience   200000 non-null  str    
 4   Duration          200000 non-null  str    
 5   Channel_Used      200000 non-null  str    
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  str    
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  str    
 10  Language          200000 non-null  str    
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  str    
 15  Date              200000 non-null  str    
dtypes: float64(2), int64(4), str(10

In [4]:
# Display the first few rows of the DataFrame
df.head()

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


In [5]:
# Describes the dataset to get a statistical summary of the numerical columns
df.describe()

,Campaign_ID,Conversion_Rate,ROI,Clicks,Impressions,Engagement_Score
count,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000
mean,100000.500000,0.080070,5.002438,549.772030,5507.301520,5.494710
std,57735.171256,0.040602,1.734488,260.019056,2596.864286,2.872581
min,1.000000,0.010000,2.000000,100.000000,1000.000000,1.000000
25%,50000.750000,0.050000,3.500000,325.000000,3266.000000,3.000000
50%,100000.500000,0.080000,5.010000,550.000000,5517.500000,5.000000
75%,150000.250000,0.120000,6.510000,775.000000,7753.000000,8.000000
max,200000.000000,0.150000,8.000000,1000.000000,10000.000000,10.000000


In [6]:
# Check for null values in the dataset
print(f"Number of null values per column:\n{df.isnull().sum()}")

Number of null values per column:
Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Location            0
Language            0
Clicks              0
Impressions         0
Engagement_Score    0
Customer_Segment    0
Date                0
dtype: int64


In [7]:
# Check for duplicated rows in the dataset
print(f"Number of duplicated rows:\n{df.duplicated().sum()}")

Number of duplicated rows:
0


In [8]:
# Check if the column Campaign_ID has unique values
print(f"Does column Campaign_ID has unique values?\n{df['Campaign_ID'].is_unique}")

Does column Campaign_ID has unique values?
True


**What the numerical summary and quality checks tell us**

- **No missing values and no duplicates.** All 16 columns have 200,000 non-null values, there are 0 duplicated rows and every `Campaign_ID` is unique (1 to 200,000). No rows need to be dropped or imputed.
- **All numerical ranges are plausible:**
  - `Conversion_Rate` goes from 0.01 to 0.15 (1% to 15%), with a mean of 0.08.
  - `ROI` goes from 2.00 to 8.00, with a mean of 5.00. There are no negative values, so no campaign in the dataset loses money.
  - `Clicks` goes from 100 to 1,000 and `Impressions` from 1,000 to 10,000.
  - `Engagement_Score` is a 1 to 10 scale, with a mean of 5.49.
- **The values look evenly spread.** In every column the mean is very close to the median, and the quartiles are spaced almost equally between the minimum and the maximum. This is typical of randomly generated data and fits the idea that the dataset is synthetic.
- **`Acquisition_Cost` is missing from this summary** because it is stored as text ("$16,174.00"). It will be converted to a number in the cleaning step.

In [9]:
# Check if any row has more clicks than impressions
print(f"Rows with more clicks than impressions:\n{(df['Clicks'] > df['Impressions']).sum()}")

Rows with more clicks than impressions:
0


**Clicks vs impressions:** there are 0 rows with more clicks than impressions, so this inconsistency does not appear in the data. This matches the ranges in `describe()`: the maximum number of clicks (1,000) equals the minimum number of impressions (1,000), so clicks can never exceed impressions.

In [10]:
# Categorical columns to analyze
# There are 10 text columns, but Acquisition_Cost (a number) and Date (a date) are not categories,
# so only the remaining 8 are kept
columns = df.select_dtypes(include="str").columns
to_exclude = ["Acquisition_Cost", "Date"]
analysis_columns = [col for col in columns if col not in to_exclude]

print(f"Categorical columns to analyze:\n{analysis_columns}")

Categorical columns to analyze:
['Company', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Location', 'Language', 'Customer_Segment']


In [11]:
# Number of distinct values per categorical column
df[analysis_columns].nunique()

Company             5
Campaign_Type       5
Target_Audience     5
Duration            4
Channel_Used        6
Location            5
Language            5
Customer_Segment    5
dtype: int64

In [12]:
# Show the values and their frequency for each categorical column
for col in analysis_columns:
    print(f"--- {col} ---")         # name of the column as a header
    print(df[col].value_counts())   # show the frequency of each value
    print()                         # print a blank line for better readability

--- Company ---
Company
TechCorp               40237
Alpha Innovations      40051
DataTech Solutions     40012
NexGen Systems         39991
Innovate Industries    39709
Name: count, dtype: int64

--- Campaign_Type ---
Campaign_Type
Influencer      40169
Search          40157
Display         39987
Email           39870
Social Media    39817
Name: count, dtype: int64

--- Target_Audience ---
Target_Audience
Men 18-24      40258
Men 25-34      40023
All Ages       40019
Women 25-34    40013
Women 35-44    39687
Name: count, dtype: int64

--- Duration ---
Duration
30 days    50255
45 days    50100
60 days    49866
15 days    49779
Name: count, dtype: int64

--- Channel_Used ---
Channel_Used
Email         33599
Google Ads    33438
YouTube       33392
Instagram     33392
Website       33360
Facebook      32819
Name: count, dtype: int64

--- Location ---


Location
Miami          40269
New York       40024
Chicago        40010
Los Angeles    39947
Houston        39750
Name: count, dtype: int64

--- Language ---
Language
Mandarin    40255
Spanish     40102
German      39983
English     39896
French      39764
Name: count, dtype: int64

--- Customer_Segment ---
Customer_Segment
Foodies                40208
Tech Enthusiasts       40151
Outdoor Adventurers    40011
Health & Wellness      39888
Fashionistas           39742
Name: count, dtype: int64



**What the categorical columns tell us**

- **No typos or inconsistent labels.** Every column has a small, clean set of values (4 to 6), with no value written in two different ways. No text cleaning is needed for these columns.
- **Categories are almost perfectly balanced.** Columns with 5 values have about 40,000 rows per category (for example, `Company` ranges from 39,709 to 40,237). `Channel_Used` has 6 channels with about 33,000 rows each (32,819 to 33,599), and `Duration` has 4 values with about 50,000 each (49,779 to 50,255). Such an even split is unusual in real marketing data and is another sign that the dataset is synthetic. It also suggests that differences between groups will likely be small.
- **`Duration` only takes 4 values:** 15, 30, 45 and 60 days. It is stored as text ("30 days"). In the cleaning step it will be converted to an integer and renamed `Duration_days`, so the unit is clear from the column name.
- **`Location` contains 5 US cities**, not countries (Miami, New York, Chicago, Los Angeles, Houston), while `Language` includes Mandarin, Spanish, German, English and French. Location-based conclusions will be about cities, not countries.

## 3. Data cleaning and preprocessing

### 3.1 Fixing column types

The first inspection showed that the data has no missing values, no duplicates and no inconsistent labels. The only problems are **three columns stored with the wrong type**:

| Column | Problem | Decision |
|---|---|---|
| `Acquisition_Cost` | Text with a dollar sign and commas (`"$16,174.00"`), so it cannot be used in calculations | Remove `$` and `,` and convert to float (USD) |
| `Duration` | Text with the unit inside the value (`"30 days"`) | Keep only the number, convert to integer and rename to `Duration_days` so the unit is in the column name |
| `Date` | Text (`"2021-01-01"`), so it cannot be used for time analysis | Convert to datetime |

**Why the cleaning code is in `src/cleaning.py` and not in this notebook:** the same cleaning steps are reused by the notebook and by the data pipeline, so they are written once as functions. Each function has tests in `tests/test_cleaning.py` that were written before the function itself, checking for example that `"$16,174.00"` becomes `16174.0`. To run them, use `pytest` from the repo root.

In [13]:
import sys

# The notebook runs from notebooks/, so add the repo root to the path to import src/
sys.path.append("..")
from src.cleaning import clean_campaigns

# Keep the raw data in df and work on a cleaned copy
df_clean = clean_campaigns(df)
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Campaign_ID       200000 non-null  int64         
 1   Company           200000 non-null  str           
 2   Campaign_Type     200000 non-null  str           
 3   Target_Audience   200000 non-null  str           
 4   Duration_days     200000 non-null  int64         
 5   Channel_Used      200000 non-null  str           
 6   Conversion_Rate   200000 non-null  float64       
 7   Acquisition_Cost  200000 non-null  float64       
 8   ROI               200000 non-null  float64       
 9   Location          200000 non-null  str           
 10  Language          200000 non-null  str           
 11  Clicks            200000 non-null  int64         
 12  Impressions       200000 non-null  int64         
 13  Engagement_Score  200000 non-null  int64         
 14  Customer_Segmen

In [14]:
# Compare the same rows before and after cleaning
print("Before cleaning:")
print(df[["Acquisition_Cost", "Duration", "Date"]].head(3))
print("\nAfter cleaning:")
print(df_clean[["Acquisition_Cost", "Duration_days", "Date"]].head(3))

Before cleaning:
  Acquisition_Cost Duration        Date
0       $16,174.00  30 days  2021-01-01
1       $11,566.00  60 days  2021-01-02
2       $10,200.00  30 days  2021-01-03

After cleaning:
   Acquisition_Cost  Duration_days       Date
0           16174.0             30 2021-01-01
1           11566.0             60 2021-01-02
2           10200.0             30 2021-01-03


In [15]:
# Check the new columns: value ranges and date coverage
print(df_clean[["Acquisition_Cost", "Duration_days"]].describe())
print(f"\nDate range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
print(f"Distinct dates: {df_clean['Date'].nunique()}")

       Acquisition_Cost  Duration_days
count     200000.000000  200000.000000
mean       12504.393040      37.503975
std         4337.664545      16.746720
min         5000.000000      15.000000
25%         8739.750000      30.000000
50%        12496.500000      30.000000
75%        16264.000000      45.000000
max        20000.000000      60.000000

Date range: 2021-01-01 to 2021-12-31
Distinct dates: 365


**What the cleaning changed**

- **All conversions worked on every row.** `df_clean` still has 200,000 rows and 0 missing values, so no value failed to convert. The row-by-row comparison shows that `"$16,174.00"` became `16174.0`, `"30 days"` became `30` and the dates kept the same value with a datetime type.
- **`Acquisition_Cost` goes from 5,000 to 20,000 USD**, with a mean of 12,504 USD. Like the other numerical columns, the mean is very close to the median (12,496.50), which again points to evenly spread, generated data.
- **`Duration_days`** now holds the 4 numeric values 15, 30, 45 and 60, so it can be used in calculations and correlations.
- **`Date` covers the full year 2021** (2021-01-01 to 2021-12-31) with all 365 days present. Since there are 200,000 campaigns, each date has many campaigns, which makes monthly or weekly trends possible.
- **The raw data is kept in `df`** and all the analysis from here uses `df_clean`. This way the original values are always available to compare.

### 3.2 Derived metrics (feature engineering)

The dataset gives raw counts and rates, but the key questions are about **efficiency**, so new metrics are needed. The dataset does not define two of its columns, so these **assumptions** are made:

- `Conversion_Rate` is **conversions / clicks**, the usual marketing definition.
- `Acquisition_Cost` is the **total spend of the campaign** in USD. Its range (5,000 to 20,000 USD) fits a campaign budget, not a cost per customer.

| New column | Formula | What it measures |
|---|---|---|
| `CTR` | Clicks / Impressions | How attractive the ad is |
| `Conversions` | Clicks × Conversion_Rate | Estimated customers acquired |
| `CPC` | Acquisition_Cost / Clicks | Cost per click (USD) |
| `Cost_per_Conversion` | Acquisition_Cost / Conversions | Cost per acquired customer (USD), the main efficiency metric |
| `CPM` | Acquisition_Cost / Impressions × 1,000 | Cost per 1,000 impressions (USD). Normalizes reach cost so campaigns of different sizes can be compared fairly |
| `Month`, `Weekday` | From `Date` | Used for seasonality |

The functions are in `src/features.py` and tested in `tests/test_features.py`. The tests use one row with easy numbers (500 clicks, 2,000 impressions, 10% conversion, 10,000 USD) so every expected result can be checked by hand.

In [16]:
from src.features import add_features

df_clean = add_features(df_clean)
df_clean[["Clicks", "Impressions", "Conversion_Rate", "Acquisition_Cost",
          "CTR", "Conversions", "CPC", "Cost_per_Conversion", "CPM", "Month", "Weekday"]].head()

,Clicks,Impressions,Conversion_Rate,Acquisition_Cost,CTR,Conversions,CPC,Cost_per_Conversion,CPM,Month,Weekday
0,506,1922,0.04,16174.0,0.263267,20.24,31.964427,799.110672,8415.192508,1,Friday
1,116,7523,0.12,11566.0,0.015419,13.92,99.706897,830.890805,1537.418583,1,Saturday
2,584,7698,0.07,10200.0,0.075864,40.88,17.465753,249.510763,1325.019486,1,Sunday
3,217,1820,0.11,12724.0,0.119231,23.87,58.635945,533.054043,6991.208791,1,Monday
4,379,4201,0.05,16452.0,0.090217,18.95,43.408971,868.179420,3916.210426,1,Tuesday


In [17]:
# Summary of the new metrics, and how many campaigns have an unrealistic CTR
new_metrics = ["CTR", "Conversions", "CPC", "Cost_per_Conversion", "CPM"]
print(df_clean[new_metrics].describe().round(3))

high_ctr = (df_clean["CTR"] > 0.5).sum()
print(f"\nCampaigns with CTR above 50%: {high_ctr} ({high_ctr / len(df_clean):.1%})")

              CTR  Conversions         CPC  Cost_per_Conversion         CPM
count  200000.000   200000.000  200000.000           200000.000  200000.000
mean        0.140       44.023      32.008              635.329    3193.943
std         0.131       32.317      26.926              970.140    2698.980
min         0.010        1.000       5.021               34.218     501.401
25%         0.059       17.760      15.092              180.484    1508.453
50%         0.100       35.760      22.774              334.396    2269.956
75%         0.170       64.100      38.599              692.481    3837.796
max         0.992      150.000     199.960            19432.353   19896.104

Campaigns with CTR above 50%: 6166 (3.1%)


**What the new metrics tell us**

- **The metrics are correct but their scale is not realistic.** The median CTR is 10.0% and 6,166 campaigns (3.1%) have a CTR above 50%, while real ad CTRs are usually a few percent. The median CPM is 2,270 USD, while real CPMs are usually tens of dollars. This is a **data quality limitation of the synthetic dataset**: comparisons *between* groups are still valid, but absolute values should not be read as real-world benchmarks.
- **A typical campaign gets about 36 conversions** (median) at a cost of **334 USD per conversion**.
- **`Cost_per_Conversion` is strongly right-skewed**: the mean (635 USD) is almost twice the median (334 USD) and the maximum reaches 19,432 USD. A few campaigns with very few conversions make the average misleading. For this reason, the analysis will compare groups using the **median**, and a **log scale** will be used when plotting this metric.
- These extreme values are checked as outliers in the next step.

## 4. Exploratory analysis

_TODO: one subsection per key question, each with a chart and a one-line reading._

## 5. Conclusions and recommendations

_TODO: write once the analysis is done._